In [ ]:
import numpy as np

def multi_head_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray,
                         W_q: np.ndarray, W_k: np.ndarray, W_v: np.ndarray,
                         W_o: np.ndarray, num_heads: int) -> np.ndarray:
    """
    Returns projected multi-head attention outputs.
    """
    pass

In [ ]:
import numpy as np


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """
    Numerically stable softmax.
    """
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def multi_head_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    W_q: np.ndarray,
    W_k: np.ndarray,
    W_v: np.ndarray,
    W_o: np.ndarray,
    num_heads: int
) -> np.ndarray:
    """
    Returns multi-head attention output.

    Expected shapes:
        Q, K, V : (batch_size, seq_length, d_model)
        W_q     : (d_model, d_model)
        W_k     : (d_model, d_model)
        W_v     : (d_model, d_model)
        W_o     : (d_model, d_model)

    Returns:
        Output : (batch_size, seq_length, d_model)
    """

    # --------------------------------------------------
    # 1. Get dimensions
    # --------------------------------------------------

    batch_size, seq_length, d_model = Q.shape

    # d_model must be divisible by number of heads
    assert d_model % num_heads == 0, \
        "d_model must be divisible by num_heads"

    d_head = d_model // num_heads

    # --------------------------------------------------
    # 2. Linear projections
    # --------------------------------------------------

    Q_new = Q @ W_q
    K_new = K @ W_k
    V_new = V @ W_v

    # --------------------------------------------------
    # 3. Split into multiple heads
    # --------------------------------------------------

    # (batch, seq_length, d_model)
    #             ↓
    # (batch, seq_length, num_heads, d_head)

    Q_heads = Q_new.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    K_heads = K_new.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    V_heads = V_new.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    # --------------------------------------------------
    # 4. Move heads before sequence dimension
    # --------------------------------------------------

    # Before:
    # (batch, seq_length, num_heads, d_head)
    #
    # After:
    # (batch, num_heads, seq_length, d_head)

    Q_heads = Q_heads.transpose(0, 2, 1, 3)
    K_heads = K_heads.transpose(0, 2, 1, 3)
    V_heads = V_heads.transpose(0, 2, 1, 3)

    # --------------------------------------------------
    # 5. Scaled dot-product attention
    # --------------------------------------------------

    # K_heads.transpose(-2, -1):
    #
    # (batch, heads, seq_length, d_head)
    #                    ↓
    # (batch, heads, d_head, seq_length)

    scores = Q_heads @ K_heads.transpose(0, 1, 3, 2)

    # Scale by sqrt(d_head)
    scores = scores / np.sqrt(d_head)

    # --------------------------------------------------
    # 6. Softmax over keys
    # --------------------------------------------------

    attention_weights = softmax(scores, axis=-1)

    # --------------------------------------------------
    # 7. Weighted sum of V
    # --------------------------------------------------

    head_outputs = attention_weights @ V_heads

    # Shape:
    # (batch, heads, seq_length, d_head)

    # --------------------------------------------------
    # 8. Move dimensions back
    # --------------------------------------------------

    head_outputs = head_outputs.transpose(0, 2, 1, 3)

    # Shape:
    # (batch, seq_length, heads, d_head)

    # --------------------------------------------------
    # 9. Concatenate all heads
    # --------------------------------------------------

    head_outputs = head_outputs.reshape(
        batch_size, seq_length, d_model
    )

    # --------------------------------------------------
    # 10. Output projection
    # --------------------------------------------------

    output = head_outputs @ W_o

    return output

In [ ]:
import numpy as np


def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def multi_head_attention(
    Q: np.ndarray, K: np.ndarray, V: np.ndarray,
    W_q: np.ndarray, W_k: np.ndarray, W_v: np.ndarray,
    W_o: np.ndarray, num_heads: int
) -> np.ndarray:

    # 1. Linear projections
    q_new = Q @ W_q
    k_new = K @ W_k
    v_new = V @ W_v

    # 2. Calculate dimension of each head
    d_model = q_new.shape[-1]
    d_head = d_model // num_heads

    # 3. Store outputs of each head
    head_outputs = []

    # 4. Process each head
    for h in range(num_heads):

        start = h * d_head
        end = start + d_head

        # Split Q, K, V for this head
        q_head = q_new[:, :, start:end]
        k_head = k_new[:, :, start:end]
        v_head = v_new[:, :, start:end]

        # 5. Scaled dot-product attention
        scores = q_head @ np.swapaxes(k_head, -1, -2)

        scores = scores / np.sqrt(d_head)

        # 6. Softmax
        attention = softmax(scores, axis=-1)

        # 7. Weighted sum of V
        head_output = attention @ v_head

        head_outputs.append(head_output)

    # 8. Concatenate all heads
    result = np.concatenate(head_outputs, axis=-1)

    # 9. Output projection
    output = result @ W_o

    return output